# NACC NP - EDA & Preprocessing
**Target:** NPADNC Binary (0+1 - Not/Low AD, 2+3 - Intermediate/High AD)  
**Model:** OSS-20B Extraction Output - 161 reports - X features

---
**Notebook flow:**
1. Setup & Data Loading
2. Sentinel -> NaN Replacement (via Binary Matrix)
3. Target Construction & Distribution
4. Feature Definition & Encoding Groups
5. Missing Data Analysis & Value Distributions
6. Near-Zero Variance (NZV) Filter
7. Encoding
8. Class Imbalance Assessment
9. Correlation Analysis & Feature Selection
10. Continuous Feature Distributions & Statistical Tests
11. Categorical Feature Distributions & Statistical Tests
12. Export & Preprocessing Summary

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
PRIMARY_DATA_PATH  = '/N/project/ADRD/neuropathoroot/results/variable_matrix_oss-20b_primary.xlsx'
RESIDUAL_DATA_PATH = '/N/project/ADRD/neuropathoroot/results/variable_matrix_oss-20b_residual.xlsx'
OUTPUT_DIR = '/N/project/ADRD/neuropathoroot/results/eda_outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'font.size':      11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi':     600,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

C_OSS   = '#12436D'
C_QWEN  = '#801650'
C_LLAMA = '#28A197'
C_GREY  = '#505a5f'
C_LIGHT = '#b1b4b6'

def save_fig(path_no_ext, fig=None):
    """Save current figure as both PNG and SVG."""
    plt.savefig(f'{path_no_ext}.png', dpi=600, bbox_inches='tight')
    plt.savefig(f'{path_no_ext}.svg', format='svg', bbox_inches='tight')

print(f"Output directory: {OUTPUT_DIR}")

In [2]:
# Load primary sheets - Raw Values for model input, Binary Matrix for sentinel mask
df_raw = pd.read_excel(PRIMARY_DATA_PATH, sheet_name='Raw Values',    header=0)
df_bin = pd.read_excel(PRIMARY_DATA_PATH, sheet_name='Binary Matrix', header=0)

# Binary Matrix has 2 trailing summary rows (N Present / % Present) - drop them
df_bin = df_bin[~df_bin['report_id'].astype(str).str.contains('Present', na=False)].reset_index(drop=True)

# Drop total_informative column - not a variable
if 'total_informative' in df_bin.columns:
    df_bin = df_bin.drop(columns=['total_informative'])

print(f"Primary Raw Values shape:    {df_raw.shape}  ({df_raw.shape[0]} reports, {df_raw.shape[1]-1} variables)")
print(f"Primary Binary Matrix shape: {df_bin.shape}  ({df_bin.shape[0]} reports, {df_bin.shape[1]-1} variables)")
assert list(df_raw.columns) == list(df_bin.columns), "Primary column mismatch between sheets!"
print("Primary column alignment confirmed")

In [3]:
# Load residual sheets - Raw Values for model input, Binary Matrix for sentinel mask
df_res_raw = pd.read_excel(RESIDUAL_DATA_PATH, sheet_name='Raw Values',    header=0)
df_res_bin = pd.read_excel(RESIDUAL_DATA_PATH, sheet_name='Binary Matrix', header=0)

# Binary Matrix has 2 trailing summary rows (N Present / % Present) - drop them
df_res_bin = df_res_bin[~df_res_bin['report_id'].astype(str).str.contains('Present', na=False)].reset_index(drop=True)

# Drop total_informative column - not a variable
if 'total_informative' in df_res_bin.columns:
    df_res_bin = df_res_bin.drop(columns=['total_informative'])

print(f"Residual Raw Values shape:    {df_res_raw.shape}  ({df_res_raw.shape[0]} reports, {df_res_raw.shape[1]-1} variables)")
print(f"Residual Binary Matrix shape: {df_res_bin.shape}  ({df_res_bin.shape[0]} reports, {df_res_bin.shape[1]-1} variables)")
assert list(df_res_raw.columns) == list(df_res_bin.columns), "Residual column mismatch between sheets!"
print("Residual column alignment confirmed")

## 2. Sentinel -> NaN Replacement (via Binary Matrix)

In [4]:
# ── Sentinel -> NaN using Binary Matrix ────────────────────────────────────────
# Binary Matrix encodes: 1 = informative value, 0 = sentinel or none
# Wherever Binary Matrix is 0 for a variable, Raw Values entry becomes NaN.

PRIMARY_VARS = [c for c in df_raw.columns if c != 'report_id']

df_sentinel = df_raw.copy()
replaced_counts = {}

for var in PRIMARY_VARS:
    # Align on report_id
    mask = df_bin.set_index('report_id')[var].reindex(df_sentinel['report_id'].values).values == 0
    n_replaced = mask.sum()
    if n_replaced > 0:
        df_sentinel.loc[mask, var] = np.nan
        replaced_counts[var] = int(n_replaced)

# ── Fix dtypes after sentinel replacement ──────────────────────────────────────
# Columns with NaN get cast to float by pandas - convert back to nullable integer
# CONTINUOUS cols stay as float
for col in PRIMARY_VARS:
    try:
        if df_sentinel[col].dropna().apply(lambda x: float(x) == int(float(x))).all():
            df_sentinel[col] = df_sentinel[col].astype('Int64')
    except (ValueError, TypeError):
        pass

print(f"Variables with at least one sentinel/none replaced: {len(replaced_counts)}")
print(f"\nTop 15 by replacements made:")
for var, n in sorted(replaced_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  {var:15s}: {n:3d} values -> NaN")

In [5]:
# ── Merge residual variables into df_sentinel ──────────────────────────────────
RESIDUAL_VARS = [c for c in df_res_raw.columns if c != 'report_id']

df_res_sentinel = df_res_raw.copy()
for var in RESIDUAL_VARS:
    mask = df_res_bin.set_index('report_id')[var].reindex(df_res_sentinel['report_id'].values).values == 0
    if mask.sum() > 0:
        df_res_sentinel.loc[mask, var] = np.nan

# Int64 coercion for residual ordinal vars (weights/mm stay float)
for col in RESIDUAL_VARS:
    try:
        if df_res_sentinel[col].dropna().apply(lambda x: float(x) == int(float(x))).all():
            df_res_sentinel[col] = df_res_sentinel[col].astype('Int64')
    except (ValueError, TypeError):
        pass

df_sentinel = df_sentinel.merge(
    df_res_sentinel[['report_id'] + RESIDUAL_VARS],
    on='report_id', how='left'
)
print(f"df_sentinel shape after residual merge: {df_sentinel.shape}")

In [6]:
# ── Export raw merged matrix (pre-filtering) for nested model pipeline ────────
# Sentinel->NaN already applied above. No missingness/NZV/correlation filtering
# yet - that happens inside each CV fold in the model notebook to avoid
# feature-selection leakage into the performance estimate.

TARGET_RAW = 'NPADNC'
TARGET_BIN = 'NPADNC_bin'

df_sentinel[TARGET_BIN] = df_sentinel[TARGET_RAW].map({0: 0, 1: 0, 2: 1, 3: 1}).astype('Int64')

raw_export_path = f'{OUTPUT_DIR}/dataset_raw_merged.csv'
df_sentinel.to_csv(raw_export_path, index=False)
print(f"Raw merged dataset saved to: {raw_export_path}")
print(f"  Shape: {df_sentinel.shape}")
print(f"  Columns: report_id, {TARGET_RAW}, {TARGET_BIN}, + {df_sentinel.shape[1]-3} candidate features")

## 3. Target Construction & Distribution

In [7]:
# NPADNC: 0=Not AD, 1=Low, 2=Intermediate, 3=High
# Binary collapse per NIA-AA guidelines (Hyman et al. 2012, PMC3266529)
# 0+1 -> 0 (Not/Low),  2+3 -> 1 (Intermediate/High)

df_sentinel[TARGET_BIN] = df_sentinel[TARGET_RAW].map({0: 0, 1: 0, 2: 1, 3: 1}).astype('Int64')

raw_counts = df_sentinel[TARGET_RAW].value_counts().sort_index()
bin_counts = df_sentinel[TARGET_BIN].value_counts().sort_index()

print("NPADNC raw distribution:")
labels_raw = {0: 'Not AD (0)', 1: 'Low (1)', 2: 'Intermediate (2)', 3: 'High (3)'}
for k, v in raw_counts.items():
    print(f"  {labels_raw[k]}: {v} ({v/len(df_sentinel)*100:.1f}%)")

print("\nNPADNC binary distribution:")
labels_bin = {0: 'Not/Low AD (0)', 1: 'Intermediate/High AD (1)'}
for k, v in bin_counts.items():
    print(f"  {labels_bin[k]}: {v} ({v/len(df_sentinel)*100:.1f}%)")

In [8]:
# ── Figure 1: Target Distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Raw 4-class
ax = axes[0]
colors_raw = [C_LIGHT, C_GREY, '#2171b5', C_OSS]
bars = ax.bar(
    [labels_raw[k] for k in raw_counts.index],
    raw_counts.values,
    color=colors_raw, edgecolor='white', linewidth=0.8
)
for bar, val in zip(bars, raw_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'n={val}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('NPADNC - Raw (4 levels)', fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, max(raw_counts.values) * 1.18)
ax.tick_params(axis='x', labelsize=9)

# Binary collapsed
ax = axes[1]
colors_bin = [C_GREY, C_OSS]
bars = ax.bar(
    [labels_bin[k] for k in bin_counts.index],
    bin_counts.values,
    color=colors_bin, edgecolor='white', linewidth=0.8, width=0.45
)
for bar, val in zip(bars, bin_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'n={val}\n({val/len(df_sentinel)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('NPADNC - Binary Collapsed', fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, max(bin_counts.values) * 1.22)
ax.tick_params(axis='x', labelsize=10)

fig.suptitle('Target Variable Distribution  (N=161)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig01_target_distribution')
plt.show()
print("Saved: fig01_target_distribution")

## 4. Feature Definition & Encoding Groups

In [9]:
# ── Feature list — Full set (109 + 53 variables) ────────────────────────────────────

FEATURE_VARS = {
    # Specimen Info
    'NPSEX':     'BINARY',
    'NACCDAGE':  'CONTINUOUS',
    'NPWBRWT':   'CONTINUOUS',
    # Gross Exam
    'NPGRCCA':   'ORDINAL',
    'NPGRLA':    'BINARY',
    'NPGRHA':    'ORDINAL',
    'NPGRSNH':   'ORDINAL',
    'NPGRLCH':   'ORDINAL',
    'NACCAVAS':  'ORDINAL',
    'NPWMR':     'ORDINAL',
    'NACCARTE':  'ORDINAL',
    # AD Pathology
    'NACCAMY':   'ORDINAL',
    # Lewy Body
    'NPLBOD':    ('ONEHOT', 0),
    # Microscopic
    'NPNLOSS':   'ORDINAL',
    'NPHIPSCL':  ('ONEHOT', 0),
    'NPSCL':     'BINARY',
    # Infarcts
    'NPLINF':    'BINARY',
    'NPLAC':     'BINARY',
    'NPINF':     'BINARY',
    'NPINF1A':   'CONTINUOUS',
    'NPINF2A':   'CONTINUOUS',
    'NPINF3A':   'CONTINUOUS',
    'NPINF4A':   'CONTINUOUS',
    # Hemorrhage
    'NPHEMO':    'BINARY',
    'NPHEMO1':   'BINARY',
    'NPHEMO2':   'BINARY',
    'NPHEMO3':   'BINARY',
    # Microinfarcts
    'NPOLD':     'BINARY',
    'NPOLD1':    'ORDINAL',
    'NPOLD2':    'ORDINAL',
    'NPOLD3':    'ORDINAL',
    'NPOLD4':    'ORDINAL',
    # Microbleeds
    'NPOLDD':    'BINARY',
    'NPOLDD1':   'ORDINAL',
    'NPOLDD2':   'ORDINAL',
    'NPOLDD3':   'ORDINAL',
    'NPOLDD4':   'ORDINAL',
    # Other Vascular
    'NPMICRO':   'BINARY',
    'NPART':     'BINARY',
    'NPOANG':    'BINARY',
    'NPPATH':    'BINARY',
    'NACCNEC':   'BINARY',
    'NPPATH7':   'BINARY',
    'NPPATH8':   'BINARY',
    'NPPATH9':   'BINARY',
    'NPPATH10':  'BINARY',
    'NPPATH11':  'BINARY',
    'NPPATHO':   'BINARY',
    # FTLD Tau
    'NACCPICK':  'BINARY',
    'NPFTDT2':   'BINARY',
    'NACCCBD':   'BINARY',
    'NACCPROG':  'BINARY',
    'NPFTDT5':   'BINARY',
    'NPFTDT6':   'BINARY',
    'NPFTDT7':   'BINARY',
    'NPFTDT8':   'BINARY',
    'NPFTDT9':   'BINARY',
    'NPFTDT10':  'BINARY',
    # FTLD General
    'NPFTDTDP':  'BINARY',
    'NPALSMND':  ('ONEHOT', 0),
    'NPOFTD1':   'BINARY',
    'NPOFTD2':   'BINARY',
    'NPOFTD3':   'BINARY',
    'NPOFTD4':   'BINARY',
    'NPOFTD5':   'BINARY',
    'NPFTDNO':   'BINARY',
    'NPFTDSPC':  'BINARY',
    'NPFTD':     ('ONEHOT', 3),
    'NPTAU':     'BINARY',
    'NPFRONT':   'BINARY',
    # TDP-43
    'NPTDPA':    'BINARY',
    'NPTDPB':    'BINARY',
    'NPTDPC':    'BINARY',
    'NPTDPD':    'BINARY',
    'NPTDPE':    'BINARY',
    # Diagnoses
    'NPPLEWY':   'BINARY',
    'NPCLEWY':   'BINARY',
    'NPPVASC':   'BINARY',
    'NPCVASC':   'BINARY',
    'NPPFTLD':   'BINARY',
    'NPCFTLD':   'BINARY',
    'NPPHIPP':   'BINARY',
    'NPCHIPP':   'BINARY',
    'NPPPRION':  'BINARY',
    'NPCPRION':  'BINARY',
    'NPPOTH1':   'BINARY',
    'NPCOTH1':   'BINARY',
    'NPPOTH2':   'BINARY',
    'NPCOTH2':   'BINARY',
    'NPPOTH3':   'BINARY',
    'NPCOTH3':   'BINARY',
    'NACCOTHP':  'BINARY',
    'NACCPRIO':  'BINARY',
    # Genetics
    'NPPDXP':    'BINARY',
    'NPPDXQ':    'BINARY',
    # Other Disease
    'NPPDXA':    'BINARY',
    'NPPDXB':    'BINARY',
    'NPPDXD':    'BINARY',
    'NPPDXE':    'BINARY',
    'NPPDXF':    'BINARY',
    'NPPDXG':    'BINARY',
    'NPPDXH':    'BINARY',
    'NPPDXI':    'BINARY',
    'NPPDXJ':    'BINARY',
    'NPPDXK':    'BINARY',
    'NPPDXL':    'BINARY',
    'NPPDXM':    'BINARY',
    'NPPDXN':    'BINARY',
    # Criteria & Misc
    'NPVOTH':    'BINARY',

    # ── Residual: Hemibrain Weights ────────────────────────────────────────────
    'hemibrain_weight_right_fresh_g':  'CONTINUOUS',
    'hemibrain_weight_left_fresh_g':   'CONTINUOUS',
    'hemibrain_weight_right_fixed_g':  'CONTINUOUS',
    'hemibrain_weight_left_fixed_g':   'CONTINUOUS',
    # Residual: Cerebral Weights
    'cerebral_weight_right_fresh_g':   'CONTINUOUS',
    'cerebral_weight_left_fresh_g':    'CONTINUOUS',
    'cerebral_weight_right_fixed_g':   'CONTINUOUS',
    'cerebral_weight_left_fixed_g':    'CONTINUOUS',
    # Residual: Cerebellar Weights
    'cerebellar_weight_right_fresh_g': 'CONTINUOUS',
    'cerebellar_weight_left_fresh_g':  'CONTINUOUS',
    'cerebellar_weight_right_fixed_g': 'CONTINUOUS',
    'cerebellar_weight_left_fixed_g':  'CONTINUOUS',
    # Residual: Brainstem Weights
    'brainstem_weight_right_fresh_g':  'CONTINUOUS',
    'brainstem_weight_left_fresh_g':   'CONTINUOUS',
    'brainstem_weight_right_fixed_g':  'CONTINUOUS',
    'brainstem_weight_left_fixed_g':   'CONTINUOUS',
    # Residual: Weight Deltas (right − left, programmatic)
    'hemibrain_weight_delta_g':        'CONTINUOUS',
    'cerebral_weight_delta_g':         'CONTINUOUS',
    'cerebellar_weight_delta_g':       'CONTINUOUS',
    'brainstem_weight_delta_g':        'CONTINUOUS',
    # Residual: Corpus Callosum
    'corpus_callosum_genu_mm':              'CONTINUOUS',
    'corpus_callosum_body_anterior_mm':     'CONTINUOUS',
    'corpus_callosum_body_mid_mm':          'CONTINUOUS',
    'corpus_callosum_body_posterior_mm':    'CONTINUOUS',
    'corpus_callosum_splenium_mm':          'CONTINUOUS',
    # Residual: Circle of Willis
    'cow_basilar_mm':        'CONTINUOUS',
    'cow_vertebral_right_mm':'CONTINUOUS',
    'cow_vertebral_left_mm': 'CONTINUOUS',
    'cow_ica_right_mm':      'CONTINUOUS',
    'cow_ica_left_mm':       'CONTINUOUS',
    'cow_mca_right_mm':      'CONTINUOUS',
    'cow_mca_left_mm':       'CONTINUOUS',
    'cow_aca_right_mm':      'CONTINUOUS',
    'cow_aca_left_mm':       'CONTINUOUS',
    'cow_pca_right_mm':      'CONTINUOUS',
    'cow_pca_left_mm':       'CONTINUOUS',
    'cow_pcom_right_mm':     'CONTINUOUS',
    'cow_pcom_left_mm':      'CONTINUOUS',
    'cow_acom_mm':           'CONTINUOUS',
    # Residual: Structural Measurements
    'caudate_nucleus_head_width_mm':        'CONTINUOUS',
    # Residual: Structural Severity (ordinal 0-3, 8=not assessed)
    'lateral_ventricle_enlargement_severity': 'ORDINAL',
    'amygdala_atrophy_severity':              'ORDINAL',
    'dilated_perivascular_spaces':            'ORDINAL',
    'cerebellar_atrophy_severity':            'ORDINAL',
    'globus_pallidus_neuronal_loss_severity': 'ORDINAL',
    'basal_ganglia_atrophy':                  'ORDINAL',
    'thalamic_degeneration_severity':         'ORDINAL',
    'brainstem_atrophy_severity':             'ORDINAL',
    # Residual: Regional Pathology (ordinal 0-3)
    'purkinje_cell_loss_severity':            'ORDINAL',
    'dentate_nucleus_atrophy':                'ORDINAL',
    'artag_severity':                         'ORDINAL',
    'gvd_hippocampus_severity':               'ORDINAL',
    'hirano_bodies_hippocampus_severity':     'ORDINAL',
}

# Verify all selected vars exist in the data
missing = [v for v in FEATURE_VARS if v not in df_sentinel.columns]
if missing:
    print(f"WARNING: {len(missing)} vars not found in data: {missing}")
else:
    print(f"Feature set: {len(FEATURE_VARS)} variables")
    for v, t in FEATURE_VARS.items():
        enc = t[0] if isinstance(t, tuple) else t
        print(f"  {v:12s}: {enc}")

In [10]:
# ── Encoding group definitions ─────────────────────────────────────────────

# Derive groups
def enc_type(t):
    return t[0] if isinstance(t, tuple) else t

CONTINUOUS   = [v for v, t in FEATURE_VARS.items() if enc_type(t) == 'CONTINUOUS']
ORDINAL      = [v for v, t in FEATURE_VARS.items() if enc_type(t) == 'ORDINAL']
ONEHOT_VARS  = [v for v, t in FEATURE_VARS.items() if enc_type(t) == 'ONEHOT']
BINARY_FLAGS = [v for v, t in FEATURE_VARS.items() if enc_type(t) == 'BINARY']

# ONEHOT reference categories
ONEHOT_REF   = {v: t[1] for v, t in FEATURE_VARS.items() if isinstance(t, tuple) and t[0] == 'ONEHOT'}

print(f"CONTINUOUS:   {CONTINUOUS}")
print(f"ORDINAL:      {ORDINAL}")
print(f"ONE-HOT:      {ONEHOT_VARS}")
print(f"BINARY:       {BINARY_FLAGS}")

## 5. Missing Data Analysis & Value Distributions

In [11]:
# Missing data across the feature columns (after sentinel replacement)
feat_cols = list(FEATURE_VARS.keys())
primary_feat_cols  = [c for c in feat_cols if c not in RESIDUAL_VARS]
residual_feat_cols = [c for c in feat_cols if c in RESIDUAL_VARS]

miss_n   = df_sentinel[feat_cols].isnull().sum().sort_values(ascending=False)
miss_pct = (miss_n / len(df_sentinel) * 100).round(1)

# Threshold buckets
def miss_bucket(p):
    if p == 0:     return 'Complete'
    elif p < 5:    return '<5%'
    elif p < 20:   return '5-20%'
    elif p < 50:   return '20-50%'
    else:          return '>50%'

def print_missing(cols, label):
    miss_n   = df_sentinel[cols].isnull().sum().sort_values(ascending=False)
    miss_pct = (miss_n / len(df_sentinel) * 100).round(1)
    miss_df  = pd.DataFrame({'Missing N': miss_n, 'Missing %': miss_pct})
    miss_df  = miss_df[miss_df['Missing N'] > 0]
    miss_df['Bucket'] = miss_df['Missing %'].apply(miss_bucket)
    print("\n")
    print(f"── {label} ({len(cols)} vars) ───────────────────────────────")

    print(f"Features with ANY missingness: {len(miss_df)} / {len(cols)}")
    print(f"Features fully complete:       {len(cols) - len(miss_df)}")
    print()
    print("Missing data by bucket:")
    for bucket in ['<5%', '5-20%', '20-50%', '>50%']:
        sub = miss_df[miss_df['Bucket'] == bucket]
        if len(sub):
            print(f"  {bucket:8s}: {len(sub):2d} vars  -> {sub.index.tolist()}")

print_missing(primary_feat_cols,  "Primary Variables")
print_missing(residual_feat_cols, "Residual Variables")

In [12]:
# ── Figure 2: Missing Data Heatmap ────────────────────────────────────────────
# Sort reports by total missingness, features by missingness %
def plot_missing_heatmap(cols, label, fig_suffix):
    m_n         = df_sentinel[cols].isnull().sum().sort_values(ascending=False)
    miss_order  = m_n[m_n > 0].index.tolist()
    n_complete  = len(cols) - len(miss_order)

    if not miss_order:
        print(f"{label}: no missing data - heatmap skipped")
        return

    df_heat  = df_sentinel[miss_order].copy()
    row_miss = df_heat.isnull().sum(axis=1).sort_values(ascending=False)
    df_heat  = df_heat.loc[row_miss.index]

    fig, ax = plt.subplots(figsize=(max(10, len(miss_order)*0.38), 7))
    sns.heatmap(
        df_heat.isnull().astype(int),
        ax=ax,
        cmap=['#e8f0f7', C_OSS],
        cbar=False,
        xticklabels=True,
        yticklabels=False,
        linewidths=0,
    )
    ax.set_title(
        f'Missing Data Heatmap - {label}\n'
        f'{len(miss_order)} features with missingness  |  {n_complete} fully complete not shown',
        fontweight='bold'
    )
    ax.set_xlabel('Feature', fontsize=10)
    ax.set_ylabel('Reports (sorted by total missingness)', fontsize=10)
    ax.tick_params(axis='x', rotation=90, labelsize=7.5)
    plt.tight_layout()
    save_fig(f'{OUTPUT_DIR}/fig02_missing_heatmap_{fig_suffix}')
    plt.show()
    print(f"Saved: fig02_missing_heatmap_{fig_suffix}")

plot_missing_heatmap(primary_feat_cols,  "Primary Variables",  "primary")
plot_missing_heatmap(residual_feat_cols, "Residual Variables", "residual")


In [13]:
# ── Figure 3: Missing % Bar Chart ─────────────────────────────────────────────
def plot_missing_bar(cols, label, fig_suffix):
    m_n   = df_sentinel[cols].isnull().sum().sort_values(ascending=False)
    m_pct = (m_n / len(df_sentinel) * 100).round(1)
    miss_df = pd.DataFrame({'Missing N': m_n, 'Missing %': m_pct})
    miss_df = miss_df[miss_df['Missing N'] > 0]

    if len(miss_df) == 0:
        print(f"{label}: no missing data - bar chart skipped")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(miss_df)*0.42), 4.5))
    colors_miss = [
        C_LLAMA  if p >= 50 else
        C_OSS    if p >= 20 else
        C_QWEN   if p >= 5  else
        C_GREY
        for p in miss_df['Missing %']
    ]
    ax.bar(miss_df.index, miss_df['Missing %'],
           color=colors_miss, edgecolor='white', linewidth=0.5)
    ax.axhline(5,  color=C_GREY, linestyle='--', linewidth=1, alpha=0.6, label='5% threshold')
    ax.axhline(20, color=C_QWEN, linestyle='--', linewidth=1, alpha=0.6, label='20% threshold')
    ax.axhline(50, color=C_OSS,  linestyle='--', linewidth=1, alpha=0.6, label='50% threshold')
    ax.set_title(f'Missing Data % per Feature - {label}\n(after sentinel -> NaN)',
                 fontweight='bold')
    ax.set_ylabel('Missing (%)')
    ax.set_xlabel('Feature')
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    save_fig(f'{OUTPUT_DIR}/fig03_missing_bar_{fig_suffix}')
    plt.show()
    print(f"Saved: fig03_missing_bar_{fig_suffix}")

plot_missing_bar(primary_feat_cols,  "Primary Variables",  "primary")
plot_missing_bar(residual_feat_cols, "Residual Variables", "residual")

In [14]:
# ── Figure 4: Value distribution for all features ─────────────────────────────
def plot_value_distributions(cols, label, fig_suffix):
    ncols   = 5
    nrows   = (len(cols) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.5, nrows*3))
    axes    = axes.flatten()

    for i, col in enumerate(cols):
        ax     = axes[i]
        series = df_sentinel[col].dropna()

        if col in CONTINUOUS:
            ax.hist(series, bins=20, color=C_OSS, alpha=0.8, edgecolor='white')
            ax.set_ylabel('Count')
        else:
            vc        = series.value_counts().sort_index()
            n_missing = df_sentinel[col].isna().sum()
            if n_missing > 0:
                vc['NA'] = n_missing
            ax.bar(vc.index.astype(str), vc.values,
                   color=[C_QWEN if str(x) == 'NA' else C_OSS for x in vc.index],
                   alpha=0.8, edgecolor='white')
            ax.set_ylabel('Count')

        ax.set_title(col, fontsize=8.5, fontweight='bold')
        ax.tick_params(labelsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'Value Distribution - {label}', fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    save_fig(f'{OUTPUT_DIR}/fig04_value_distributions_{fig_suffix}')
    plt.show()
    print(f"Saved: fig04_value_distributions_{fig_suffix}")

plot_value_distributions(primary_feat_cols,  "Primary Variables",  "primary")
plot_value_distributions(residual_feat_cols, "Residual Variables", "residual")

In [15]:
# ── Missingness summary ────────────────────────────────────────────────────────
MISS_THRESHOLD = 40

def print_miss_summary(cols, label):
    m_n   = miss_n[cols]
    m_pct = miss_pct[cols]
    w     = max((len(v) for v in cols), default=20) + 2  # align to longest var name

    low      = m_pct[(m_pct > 0) & (m_pct < 5)].sort_values(ascending=False).index.tolist()
    high     = m_pct[(m_pct >= 5) & (m_pct < MISS_THRESHOLD)].sort_values(ascending=False).index.tolist()
    flagged  = m_pct[m_pct >= MISS_THRESHOLD].sort_values(ascending=False).index.tolist()
    complete = m_pct[m_pct == 0].index.tolist()

    print(f"── {label} ({len(cols)} vars) ───────────────────────────────────────")
    print(f"  Fully complete:               {len(complete)}")
    print(f"  Low missingness   (<5%):      {len(low)}")
    print(f"  High missingness  (5-{MISS_THRESHOLD}%):    {len(high)}")
    print(f"  Flagged           (>={MISS_THRESHOLD}%):    {len(flagged)}")

    if low:
        print(f"\n  Low missingness features (<5%):")
        for v in low:
            print(f"    {v:{w}s}  missing = {m_n[v]:3d}  ({m_pct[v]:5.1f}%)")

    if high:
        print(f"\n  High missingness features (5-{MISS_THRESHOLD}%):")
        for v in high:
            print(f"    {v:{w}s}  missing = {m_n[v]:3d}  ({m_pct[v]:5.1f}%)")

    if flagged:
        print(f"\n  Flagged features (>={MISS_THRESHOLD}%) - NOT dropped here:")
        for v in flagged:
            print(f"    {v:{w}s}  missing = {m_n[v]:3d}  ({m_pct[v]:5.1f}%)")
    else:
        print(f"\n  No features above {MISS_THRESHOLD}% threshold.")
    print()

print_miss_summary(primary_feat_cols,  "Primary Variables")
print_miss_summary(residual_feat_cols, "Residual Variables")

print("Note: NaNs handled per model inside CV pipeline.")
print("XGBoost: Native NaN handling.")
print("LR/RF/SVM: SimpleImputer(most_frequent) inside pipeline.")

# ── Global flagged list (used by downstream drop cell) ────────────────────────
flagged_miss = miss_pct[miss_pct >= MISS_THRESHOLD].sort_values(ascending=False).index.tolist()
print(f"\nTotal features flagged (>={MISS_THRESHOLD}%): {len(flagged_miss)}")
print("Note: Flagged but NOT automatically dropped.")

In [16]:
# ── Remove flagged features ────────────────────────────────────────────────────
N_ORIGINAL_FEAT = len(feat_cols)
feat_cols    = [v for v in feat_cols if v not in flagged_miss]
FEATURE_VARS = {k: v for k, v in FEATURE_VARS.items() if k not in flagged_miss}
N_AFTER_MISS_DROP = len(feat_cols)

# Update partition lists
primary_feat_cols  = [c for c in feat_cols if c not in RESIDUAL_VARS]
residual_feat_cols = [c for c in feat_cols if c in RESIDUAL_VARS]
print()
print(f"Features dropped (>={MISS_THRESHOLD}% missing): {len(flagged_miss)}")
print(f"Features remaining: {len(feat_cols)}")
print(f"  Primary:  {len(primary_feat_cols)}")
print(f"  Residual: {len(residual_feat_cols)}")

## 6. Near-Zero Variance (NZV) Filter

A feature where one value dominates ≥T% of samples provides near-zero predictive signal.  
For a binary feature, dominance threshold T% corresponds to `sklearn VarianceThreshold = T*(1-T)`.

We compare **T=90%** and **T=95%** below.

In [17]:
def nzv_analysis(df_feats, threshold_pct):
    """
    Return list of columns that pass NZV filter at given dominance threshold.
    threshold_pct: e.g. 0.95 means drop if any value appears in >=95% of non-NaN rows.
    """
    dropped, kept = [], []
    detail = {}
    for col in df_feats.columns:
        series  = df_feats[col].dropna()
        if len(series) == 0:
            dropped.append(col)
            detail[col] = {'dominant_val': None, 'dominant_pct': 100.0, 'n_unique': 0}
            continue
        vc = series.value_counts(normalize=True)
        dom_pct = vc.iloc[0] * 100
        if dom_pct >= threshold_pct * 100:
            dropped.append(col)
            detail[col] = {
                'dominant_val': vc.index[0],
                'dominant_pct': round(dom_pct, 1),
                'n_unique': series.nunique()
            }
        else:
            kept.append(col)
    return kept, dropped, detail

df_feats_only = df_sentinel[feat_cols].copy()

kept_90, dropped_90, detail_90 = nzv_analysis(df_feats_only, threshold_pct=0.90)
kept_95, dropped_95, detail_95 = nzv_analysis(df_feats_only, threshold_pct=0.95)

print(f"  NZV at 90% dominance threshold:")
print(f"    Features DROPPED : {len(dropped_90)}")
print(f"    Features KEPT    : {len(kept_90)}")
print()
print(f"  NZV at 95% dominance threshold:")
print(f"    Features DROPPED : {len(dropped_95)}")
print(f"    Features KEPT    : {len(kept_95)}")
print()
diff = set(dropped_90) - set(dropped_95)
print(f"  Features dropped at 90% but NOT at 95% ({len(diff)} vars):")
for v in sorted(diff):
    d = detail_90[v]
    print(f"    {v:15s}  dominant={d['dominant_val']}  ({d['dominant_pct']}%)")

# ── Split by primary / residual ───────────────────────────────────────────────
print()
print("  ── By group ─────────────────────────────────────────────────────────")
for label, cols in [("Primary", primary_feat_cols), ("Residual", residual_feat_cols)]:
    d90 = [v for v in dropped_90 if v in cols]
    d95 = [v for v in dropped_95 if v in cols]
    k90 = [v for v in kept_90   if v in cols]
    print(f"  {label} - 90%: dropped={len(d90)}, kept={len(k90)}  |  "
          f"95%: dropped={len(d95)}, kept={len([v for v in kept_95 if v in cols])}")

In [18]:
# ── Figure 5: NZV Dominance Distribution ──────────────────────────────────────
def plot_nzv_dominance(cols, label, ax):
    dom_pcts = {}
    for col in cols:
        series = df_feats_only[col].dropna()
        if len(series) > 0:
            dom_pcts[col] = series.value_counts(normalize=True).iloc[0] * 100
    dom_series = pd.Series(dom_pcts).sort_values(ascending=False)

    colors_dom = [
        C_OSS   if p >= 95 else
        C_QWEN  if p >= 90 else
        C_LLAMA if p >= 80 else
        C_GREY
        for p in dom_series.values
    ]
    ax.bar(dom_series.index, dom_series.values, color=colors_dom,
           edgecolor='white', linewidth=0.4)
    ax.axhline(95, color=C_OSS,  linestyle='--', linewidth=1.5, label='95% threshold')
    ax.axhline(90, color=C_QWEN, linestyle='--', linewidth=1.5, label='90% threshold')
    ax.set_title(f'Dominant-Value % - {label}', fontweight='bold')
    ax.set_ylabel('Dominant value prevalence (%)')
    ax.set_xlabel(f'Features (n={len(dom_series)})')
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    ax.legend(fontsize=9)

    n_above95 = (dom_series >= 95).sum()
    n_above90 = (dom_series >= 90).sum()
    ax.annotate(f'{n_above95} vars ≥95%', xy=(dom_series.index[-1], 96),
                ha='right', fontsize=9, color=C_OSS, fontweight='bold')
    ax.annotate(f'{n_above90} vars ≥90%', xy=(dom_series.index[-1], 91),
                ha='right', fontsize=9, color=C_QWEN, fontweight='bold')

primary_surviving  = [c for c in primary_feat_cols  if c in df_feats_only.columns]
residual_surviving = [c for c in residual_feat_cols if c in df_feats_only.columns]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9))
plot_nzv_dominance(primary_surviving,  "Primary Variables",  ax1)
plot_nzv_dominance(residual_surviving, "Residual Variables", ax2)

plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig05_nzv_dominance')
plt.show()
print("Saved: fig05_nzv_dominance")

In [19]:
# ── DECISION CELL ─────────────────────────────────────────────────────────────
# Set threshold here
# Options: 0.90 or 0.95

NZV_THRESHOLD = 0.95

if NZV_THRESHOLD == 0.95:
    KEPT_FEATURES, DROPPED_NZV = kept_95, dropped_95
    detail_nzv = detail_95
else:
    KEPT_FEATURES, DROPPED_NZV = kept_90, dropped_90
    detail_nzv = detail_90

print(f"NZV threshold selected: {int(NZV_THRESHOLD*100)}%")
print(f"Features after NZV filter: {len(KEPT_FEATURES)} / {len(FEATURE_VARS)}")
print(f"Dropped: {DROPPED_NZV}")

## 7. Encoding

In [20]:
# ── Apply encoding on NZV-filtered feature set ────────────────────────────────
# Only transformation done here: one-hot expansion.
# All binary and ordinal vars passed through unchanged.

df_enc = df_sentinel[['report_id', TARGET_RAW, TARGET_BIN]].copy()

for col in KEPT_FEATURES:
    if col in ONEHOT_VARS:
        ref_val = ONEHOT_REF[col]
        dummies = pd.get_dummies(df_sentinel[col], prefix=col)
        # Drop the reference category column to avoid dummy trap
        ref_col = f"{col}_{ref_val}"
        if ref_col in dummies.columns:
            dummies.drop(columns=[ref_col], inplace=True)
        for c in dummies.columns:
            df_enc[c] = dummies[c].astype('Int64')
    else:
        # Binary (0/1 or 1/2), ordinal, continuous - all passed through unchanged
        df_enc[col] = df_sentinel[col].copy()

print(f"Encoded dataframe shape: {df_enc.shape}")
print(f"Feature columns before one-hot expansion: {len(KEPT_FEATURES)}")
print(f"Feature columns after one-hot expansion: {df_enc.shape[1] - 3}")
df_enc.head(3)

## 8. Class Imbalance Assessment

In [21]:
bin_counts = df_enc[TARGET_BIN].value_counts().sort_index()
n_class0   = bin_counts[0]
n_class1   = bin_counts[1]
ratio      = n_class0 / n_class1

print(f"Class 0 (Not/Low AD):            {n_class0} ({n_class0/len(df_enc)*100:.1f}%)")
print(f"Class 1 (Intermediate/High AD):  {n_class1} ({n_class1/len(df_enc)*100:.1f}%)")
print(f"Imbalance ratio (0:1):           {ratio:.2f}:1")
print()

if ratio < 1.5:
    print("Assessment: BALANCED - no special handling needed")
elif ratio < 3.0:
    print("Assessment: MILD IMBALANCE - use class_weight='balanced' in all sklearn models")
elif ratio < 5.0:
    print("Assessment: MODERATE IMBALANCE - class_weight='balanced' required; consider SMOTE on train fold only")
else:
    print("Assessment: SEVERE IMBALANCE - SMOTE + class_weight strongly recommended")

## 9. Correlation Analysis & Feature Selection

In [22]:
# ── Figure 6: Correlation heatmap (Spearman) ──────────────────────────────────
# Use Spearman - appropriate for mixed ordinal/binary/continuous data

enc_feat_cols = [c for c in df_enc.columns
                 if c not in ['report_id', TARGET_RAW, TARGET_BIN]]
feat_for_corr = [c for c in enc_feat_cols if df_enc[c].nunique() > 1]
corr_matrix = df_enc[feat_for_corr].corr(method='spearman')

# Identify highly correlated pairs (|r| > 0.85)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.85:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

w = max((len(a) for a, b, r in high_corr_pairs), default=20) + 2
print(f"Feature pairs with |Spearman r| > 0.85: {len(high_corr_pairs)}")
for a, b, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
    print(f"  {a:{w}s} <-> {b:{w}s}  r = {r:+.3f}")

corr_matrix.to_csv(f'{OUTPUT_DIR}/spearman_feature_correlation.csv')

In [23]:
# ── Figure 6: Correlation heatmap (Spearman) ──────────────────────────────────
def plot_corr_heatmap(cols, label, fig_suffix):
    feat_cols_corr = [c for c in cols if c in df_enc.columns and df_enc[c].nunique() > 1]
    if len(feat_cols_corr) < 2:
        print(f"{label}: not enough features for correlation — skipped")
        return
    corr = df_enc[feat_cols_corr].corr(method='spearman')
    size = max(10, len(feat_cols_corr) * 0.35)
    fig, ax = plt.subplots(figsize=(size, size))
    sns.heatmap(
        corr,
        ax=ax,
        cmap='RdBu_r',
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0,
        cbar_kws={'shrink': 0.6, 'label': 'Spearman r'},
        xticklabels=True,
        yticklabels=True,
    )
    ax.set_title(f'Spearman Correlation Matrix - {label}',
                 fontweight='bold', fontsize=12)
    ax.tick_params(axis='x', rotation=90, labelsize=6)
    ax.tick_params(axis='y', rotation=0,  labelsize=6)
    plt.tight_layout()
    save_fig(f'{OUTPUT_DIR}/fig06_correlation_matrix_{fig_suffix}')
    plt.show()
    print(f"Saved: fig06_correlation_matrix_{fig_suffix}")

primary_enc  = [c for c in primary_feat_cols  if c in feat_for_corr]
residual_enc = [c for c in residual_feat_cols if c in feat_for_corr]

plot_corr_heatmap(primary_enc,  "Primary Variables",  "primary")
plot_corr_heatmap(residual_enc, "Residual Variables", "residual")
plot_corr_heatmap(feat_for_corr, "All Variables",     "combined")

In [24]:
# ── Figure 7: Point-biserial / Spearman correlation with TARGET ───────────────
# How strongly does each feature correlate with NPADNC_bin?
target_corr = {}
for col in feat_for_corr:
    paired = df_enc[[col, TARGET_BIN]].dropna()
    if len(paired) > 10 and paired[col].nunique() > 1:
        r, p = stats.spearmanr(paired[col], paired[TARGET_BIN])
        target_corr[col] = {'r': round(r, 3), 'p': round(p, 4)}

corr_target_df = pd.DataFrame(target_corr).T
corr_target_df = corr_target_df.sort_values('r', ascending=False)

def plot_target_corr(cols, label, fig_suffix):
    sub = corr_target_df[corr_target_df.index.isin(cols)]
    if len(sub) == 0:
        print(f"{label}: no features — skipped")
        return
    fig, ax = plt.subplots(figsize=(max(10, len(sub)*0.38), 5))
    colors_tc = [C_OSS if r > 0 else C_QWEN for r in sub['r']]
    ax.bar(sub.index, sub['r'], color=colors_tc,
           edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axhline(0,    color='black', linewidth=0.8)
    ax.axhline( 0.3, color=C_OSS,  linestyle='--', linewidth=1, alpha=0.5)
    ax.axhline(-0.3, color=C_QWEN, linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(f'Spearman Correlation with Target - {label}'
                 '\nDashed lines = |r|=0.3 reference', fontweight='bold')
    ax.set_ylabel('Spearman r')
    ax.set_xlabel('Feature')
    ax.tick_params(axis='x', rotation=90, labelsize=7.5)
    plt.tight_layout()
    save_fig(f'{OUTPUT_DIR}/fig07_target_correlation_{fig_suffix}')
    plt.show()
    print(f"Saved: fig07_target_correlation_{fig_suffix}")

primary_corr  = [c for c in primary_feat_cols  if c in corr_target_df.index]
residual_corr = [c for c in residual_feat_cols if c in corr_target_df.index]

plot_target_corr(primary_corr,  "Primary Variables",  "primary")
plot_target_corr(residual_corr, "Residual Variables", "residual")

# ── Combined printout and export ───────────────────────────────────────────────
print(f"\nTop 10 features by |r| with target:")
print(corr_target_df.reindex(
    corr_target_df['r'].abs().sort_values(ascending=False).index
).head(20).to_string())

corr_target_df.to_csv(f'{OUTPUT_DIR}/spearman_target_correlation.csv')

In [25]:
# ── Correlation-based feature removal ─────────────────────────────────────────
# For any pair with |Spearman r| > 0.85, drop the one with lower target correlation

DROP_CORR_FEATURES = False

TO_DROP_CORR = []

for a, b, r in high_corr_pairs:
    r_a = abs(target_corr.get(a, {}).get('r', 0))
    r_b = abs(target_corr.get(b, {}).get('r', 0))
    drop = b if r_a >= r_b else a
    keep = a if drop == b else b

    if DROP_CORR_FEATURES:
        if drop not in TO_DROP_CORR:
            TO_DROP_CORR.append(drop)
            print(
                f"  Dropping {drop} (r_target={min(r_a, r_b):.3f}) — correlated with "
                f"{keep} (r_target={max(r_a, r_b):.3f}), pair r={r:+.3f}"
            )
    else:
        print(
            f"  Keeping both {a} and {b} — pair r={r:+.3f}, "
            f"target r: {a}={r_a:.3f}, {b}={r_b:.3f}"
        )

FINAL_FEATURE_COLS = [v for v in enc_feat_cols if v not in TO_DROP_CORR]

if DROP_CORR_FEATURES:
    df_enc = df_enc[
        ['report_id', TARGET_RAW, TARGET_BIN] +
        FINAL_FEATURE_COLS
    ].copy()

print(f"\nFeatures after correlation handling: {len(FINAL_FEATURE_COLS)}")
print(f"Dropped: {TO_DROP_CORR if TO_DROP_CORR else 'None'}")

## 10. Continuous Feature Distributions & Statistical Tests

In [26]:
# ── Statistical test selector ──────────────────────────────────────────────────

from scipy.stats import shapiro, ttest_ind, mannwhitneyu, chi2_contingency, fisher_exact

def get_pvalue(col, s0, s1, feature_type):
    """Select appropriate statistical test based on feature type and data properties."""
    if feature_type == 'CONTINUOUS':
        _, p0 = shapiro(s0) if len(s0) >= 3 else (None, 0)
        _, p1 = shapiro(s1) if len(s1) >= 3 else (None, 0)
        if p0 > 0.05 and p1 > 0.05:
            _, pval = ttest_ind(s0, s1)
            test = 't-test'
        else:
            _, pval = mannwhitneyu(s0, s1, alternative='two-sided')
            test = 'Wilcoxon'
    else:
        all_vals = sorted(set(s0.tolist() + s1.tolist()))
        table = [[sum(s0 == v) for v in all_vals],
                 [sum(s1 == v) for v in all_vals]]
        table_arr = np.array(table)
        row_sums = table_arr.sum(axis=1, keepdims=True)
        col_sums = table_arr.sum(axis=0, keepdims=True)
        expected = row_sums * col_sums / table_arr.sum()
        if expected.min() >= 5:
            _, pval, _, _ = chi2_contingency(table_arr)
            test = 'Chi2'
        else:
            if table_arr.shape == (2, 2):
                _, pval = fisher_exact(table_arr)
                test = 'Fisher'
            else:
                _, pval, _, _ = chi2_contingency(table_arr)
                test = 'Chi2*'
    return pval, test

In [27]:
# ── Figure 8: Continuous feature distributions by class ───────────────────────
def plot_continuous_dists(cols, label, fig_suffix):
    cont = [c for c in cols if c in CONTINUOUS and c in FINAL_FEATURE_COLS]
    if len(cont) == 0:
        print(f"{label}: no continuous features — skipped")
        return

    fig, axes = plt.subplots(2, len(cont), figsize=(len(cont)*3.8, 8))
    if len(cont) == 1:
        axes = axes.reshape(2, 1)

    for i, col in enumerate(cont):
        series_0 = df_enc[df_enc[TARGET_BIN] == 0][col].dropna()
        series_1 = df_enc[df_enc[TARGET_BIN] == 1][col].dropna()

        ax_hist = axes[0, i]
        ax_hist.hist(series_0, bins=18, alpha=0.65, color=C_GREY, label='Not/Low AD',  density=True)
        ax_hist.hist(series_1, bins=18, alpha=0.65, color=C_OSS,  label='Int/High AD', density=True)
        ax_hist.set_title(col, fontweight='bold', fontsize=10)
        ax_hist.set_ylabel('Density' if i == 0 else '')
        if i == 0:
            ax_hist.legend(fontsize=8)

        ax_box = axes[1, i]
        ax_box.boxplot(
            [series_0.values, series_1.values],
            labels=['Not/Low AD', 'Int/High AD'],
            patch_artist=True,
            boxprops=dict(facecolor='none', color=C_GREY),
            medianprops=dict(color=C_OSS, linewidth=2),
            whiskerprops=dict(color=C_GREY),
            capprops=dict(color=C_GREY),
            flierprops=dict(markerfacecolor=C_GREY, marker='o', markersize=3, alpha=0.5),
        )
        ax_box.set_ylabel(col)
        if len(series_0) > 0 and len(series_1) > 0:
            pval, test = get_pvalue(col, series_0, series_1, 'CONTINUOUS')
            sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
            ax_box.set_xlabel(f'p={pval:.3f} {sig} ({test})', fontsize=8.5)

    fig.suptitle(f'Continuous Feature Distributions by Target Class - {label}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(f'{OUTPUT_DIR}/fig08_continuous_distributions_{fig_suffix}')
    plt.show()
    print(f"Saved: fig08_continuous_distributions_{fig_suffix}")

primary_cont  = [c for c in primary_feat_cols  if c in FINAL_FEATURE_COLS]
residual_cont = [c for c in residual_feat_cols if c in FINAL_FEATURE_COLS]

plot_continuous_dists(primary_cont,  "Primary Variables",  "primary")
plot_continuous_dists(residual_cont, "Residual Variables", "residual")

## 11. Categorical Feature Distributions & Statistical Tests

In [28]:
# ── Figure 9: Ordinal features by class ───────────────────────────────────────

ordinal_present = [c for c in ORDINAL if c in FINAL_FEATURE_COLS]
ordinal_plot = [c for c in ordinal_present if df_enc[c].nunique() > 1]

n_plot = len(ordinal_plot)
ncols = 4
nrows = (n_plot + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.8, nrows*3.5))
axes = axes.flatten() if n_plot > 1 else [axes]

for i, col in enumerate(ordinal_plot):
    ax = axes[i]
    for cls, clr, lbl in [(0, C_GREY, 'Not/Low AD'), (1, C_OSS, 'Int/High AD')]:
        subset = df_enc[df_enc[TARGET_BIN] == cls][col].dropna()
        vc     = subset.value_counts(normalize=True).sort_index() * 100
        ax.bar(vc.index.astype(float) + (0.2 if cls == 1 else -0.2),
               vc.values, width=0.38, color=clr, alpha=0.85,
               label=lbl if i == 0 else '')

    all_vals = df_enc[col].dropna().unique()
    ax.set_xticks(sorted(all_vals))
    ax.set_xticklabels([int(x) for x in sorted(all_vals)])
    ax.set_title(col, fontsize=9.5, fontweight='bold')
    ax.set_ylabel('% within class' if i % ncols == 0 else '')
    ax.tick_params(labelsize=8)

    s0 = df_enc[df_enc[TARGET_BIN] == 0][col].dropna()
    s1 = df_enc[df_enc[TARGET_BIN] == 1][col].dropna()
    if len(s0) > 0 and len(s1) > 0:
        pval, test = get_pvalue(col, s0, s1, 'ORDINAL')
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
        ax.set_xlabel(f'p={pval:.3f} {sig} ({test})', fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

handles = [
    plt.Rectangle((0,0), 1, 1, color=C_GREY, alpha=0.85),
    plt.Rectangle((0,0), 1, 1, color=C_OSS,  alpha=0.85),
]
fig.legend(handles, ['Not/Low AD', 'Int/High AD'], loc='lower right',
           fontsize=10, frameon=False)
fig.suptitle('Ordinal Feature Distributions by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(f'{OUTPUT_DIR}/fig09_ordinal_by_class')
plt.show()
print("Saved: fig09_ordinal_by_class")

In [29]:
# ── Figure 10: Binary features by class ───────────────────────────────────────

binary_present = [c for c in FINAL_FEATURE_COLS 
                  if c not in ORDINAL 
                  and c not in CONTINUOUS]
binary_plot    = [c for c in binary_present if df_enc[c].nunique() > 1]

n_plot = len(binary_plot)
ncols  = 4
nrows  = (n_plot + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.8, nrows*3.5))
axes = axes.flatten() if n_plot > 1 else [axes]

for i, col in enumerate(binary_plot):
    ax = axes[i]
    for cls, clr, lbl in [(0, C_GREY, 'Not/Low AD'), (1, C_OSS, 'Int/High AD')]:
        subset = df_enc[df_enc[TARGET_BIN] == cls][col].dropna()
        vc     = subset.value_counts(normalize=True).sort_index() * 100
        ax.bar(vc.index.astype(float) + (0.2 if cls == 1 else -0.2),
               vc.values, width=0.38, color=clr, alpha=0.85,
               label=lbl if i == 0 else '')

    all_vals = df_enc[col].dropna().unique()
    ax.set_xticks(sorted(all_vals))
    ax.set_xticklabels([int(x) for x in sorted(all_vals)])
    ax.set_title(col, fontsize=9.5, fontweight='bold')
    ax.set_ylabel('% within class' if i % ncols == 0 else '')
    ax.tick_params(labelsize=8)

    s0 = df_enc[df_enc[TARGET_BIN] == 0][col].dropna()
    s1 = df_enc[df_enc[TARGET_BIN] == 1][col].dropna()
    if len(s0) > 0 and len(s1) > 0:
        pval, test = get_pvalue(col, s0, s1, 'BINARY')
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
        ax.set_xlabel(f'p={pval:.3f} {sig} ({test})', fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

handles = [
    plt.Rectangle((0,0), 1, 1, color=C_GREY, alpha=0.85),
    plt.Rectangle((0,0), 1, 1, color=C_OSS,  alpha=0.85),
]
fig.legend(handles, ['Not/Low AD', 'Int/High AD'], loc='lower right',
           fontsize=10, frameon=False)
fig.suptitle('Binary Feature Distributions by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.98])
save_fig(f'{OUTPUT_DIR}/fig10_binary_by_class')
plt.show()
print("Saved: fig10_binary_by_class")

In [30]:
# ── Statistical significance table — all features ─────────────────────────────
print(f"{'Feature':<20} {'Type':<10} {'Test':<8} {'p-value':<12} {'Sig':<5} {'Spearman r'}")
print("-" * 68)

stats_rows = []
for col in FINAL_FEATURE_COLS:
    if col not in df_enc.columns:
        continue
    s0 = df_enc[df_enc[TARGET_BIN] == 0][col].dropna()
    s1 = df_enc[df_enc[TARGET_BIN] == 1][col].dropna()
    if len(s0) == 0 or len(s1) == 0 or s0.nunique() + s1.nunique() <= 1:
        continue
    feat_type = enc_type(FEATURE_VARS[col]) if col in FEATURE_VARS else 'ONEHOT'
    pval, test = get_pvalue(col, s0, s1, feat_type)
    sig  = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    r    = target_corr.get(col, {}).get('r', None)
    r_p  = target_corr.get(col, {}).get('p', None)
    print(f"  {col:<18} {feat_type:<10} {test:<8} {pval:<12.4f} {sig:<5} {r if r is not None else 'N/A'}")
    stats_rows.append({
        'Feature':      col,
        'Type':         feat_type,
        'Test':         test,
        'p_value':      round(pval, 4),
        'Significance': sig,
        'Spearman_r':   r,
        'Spearman_p':   r_p,
    })

stats_df = pd.DataFrame(stats_rows).set_index('Feature')
stats_df.to_csv(f'{OUTPUT_DIR}/statistical_tests.csv')
print(f"\nSaved: statistical_tests.csv")

## 12. Export & Preprocessing Summary

In [31]:
print("PREPROCESSING SUMMARY")
print(f"  Raw reports:                     {len(df_raw)}")
print(f"  Reports after preprocessing:     {len(df_enc)}")
print(f"  Original FEATURE vars:           {N_ORIGINAL_FEAT}")
print(f"  After missingness drop (>={MISS_THRESHOLD}%):  {N_AFTER_MISS_DROP}")
print(f"  After NZV filter ({int(NZV_THRESHOLD*100)}%):          {len(KEPT_FEATURES)}")
print(f"  After one-hot expansion:         {len(enc_feat_cols)}")
print(f"  Final features count:            {len(KEPT_FEATURES) - len(TO_DROP_CORR)}")
print(f"  Final model columns:             {df_enc.shape[1] - 3}")
print(f"  Target class 0 (Not/Low AD):     {(df_enc[TARGET_BIN]==0).sum()}")
print(f"  Target class 1 (Int/High AD):    {(df_enc[TARGET_BIN]==1).sum()}")
print(f"  Features with NaN remaining:     {(df_enc[FINAL_FEATURE_COLS].isnull().sum()>0).sum()}")
print(f"  (NaN retained for XGBoost native handling)")

# Save clean dataset - all columns including report_id and targets
out_path = f'{OUTPUT_DIR}/dataset_clean.csv'
df_enc.to_csv(out_path, index=False)
print(f"\nClean dataset saved to: {out_path}")

# Save final feature list - post correlation removal
feat_list_path = f'{OUTPUT_DIR}/feature_list.txt'
with open(feat_list_path, 'w') as f:
    for col in FINAL_FEATURE_COLS:
        f.write(col + '\n')
print(f"Feature list saved to:   {feat_list_path}")